# Drug Repurposing in Psychiatry — ClinicalTrials.gov + PubMed Search

**What this notebook does:**
1. Queries the ClinicalTrials.gov v2 API for psychiatric trials involving repurposed drugs
2. Flags trials against an inclusion/exclusion schema aligned with your repurposing definition
3. Queries PubMed (NCBI E-utilities) for papers linked to each NCT number
4. Exports:
   - `drug_repurposing_trials.xlsx` (two sheets)
   - `clinicaltrials_references.ris`
   - `pubmed_references.ris`

**Runtime:** ~5–15 min depending on result volume and API rate limits.

---
### Important limitations
- ClinicalTrials.gov has no native 'repurposing' filter. Classification is heuristic — **manual review required**.
- PubMed NCT linkage is incomplete: many trial results are published without the NCT indexed.
- Trials marked `UNCERTAIN` need human adjudication.

In [1]:
# Cell 1: Install dependencies
!pip install requests openpyxl tqdm --quiet

In [11]:
import requests
import time
import re
from datetime import datetime
from tqdm import tqdm
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PSYCHIATRIC_CONDITIONS = [
    "major depressive disorder", "bipolar disorder", "schizophrenia",
    "generalized anxiety disorder", "post-traumatic stress disorder",
    "obsessive compulsive disorder", "alcohol use disorder",
    "opioid use disorder", "substance use disorder", "eating disorder",
    "anorexia nervosa", "bulimia nervosa", "binge eating disorder",
    "attention deficit hyperactivity disorder", "autism spectrum disorder",
    "borderline personality disorder", "schizoaffective disorder",
    "panic disorder", "social anxiety disorder"
]

PRIMARY_PSYCH_DRUGS = {
    "sertraline","fluoxetine","escitalopram","citalopram","paroxetine",
    "venlafaxine","desvenlafaxine","duloxetine","mirtazapine","bupropion",
    "amitriptyline","nortriptyline","imipramine","clomipramine","phenelzine",
    "tranylcypromine","moclobemide","trazodone","nefazodone","vilazodone",
    "vortioxetine","levomilnacipran",
    "haloperidol","chlorpromazine","fluphenazine","perphenazine","thioridazine",
    "risperidone","olanzapine","quetiapine","aripiprazole","ziprasidone",
    "clozapine","paliperidone","asenapine","lurasidone","iloperidone",
    "brexpiprazole","cariprazine","lumateperone","pimavanserin",
    "lithium","buspirone","hydroxyzine",
    "methylphenidate","amphetamine","dextroamphetamine","lisdexamfetamine","atomoxetine",
    "naltrexone","buprenorphine","methadone","acamprosate","disulfiram","varenicline",
}

EXCLUSION_TERMS = [
    "psychotherapy","cbt","cognitive behav","mindfulness","exercise","yoga",
    "meditation","tms","transcranial","omega","vitamin","supplement","herbal",
    "sham"
]

# Add an APPROVED_DRUGS list for filtering out new, unmarketed drugs.
# This list should contain the lowercase names of FDA-approved drugs.
# To populate this, you might need to manually extract data from sources
# like https://www.accessdata.fda.gov/scripts/cder/daf/index.cfm or similar.
# For demonstration, a small placeholder list is provided.
APPROVED_DRUGS = [
    "aspirin", "metformin", "ibuprofen", "omeprazole", "lisinopril", "atorvastatin",
    "sildenafil", "tadalafil", "ketamine", "dextromethorphan", "naloxone",
    "insulin", "morphine", "acetaminophen", "warfarin"
    # ... add more approved drugs here ...
]
APPROVED_DRUGS_SET = set(d.lower() for d in APPROVED_DRUGS) # For efficient lookup

CT_BASE      = "https://clinicaltrials.gov/api/v2/studies"
PUBMED_SEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
PUBMED_FETCH  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

print(f"Configuration loaded. Searching {len(PSYCHIATRIC_CONDITIONS)} conditions.")

Configuration loaded. Searching 19 conditions.


### Upload and load Approved Drugs from Excel

Upload your Excel file containing the approved drug names. Once uploaded, provide the column name that contains the drug names.

In [2]:
from google.colab import files
import pandas as pd

print("Please upload your Excel file containing approved drug names:")
uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        print(f'User uploaded file "{filename}"')
        excel_filename = filename
        break
else:
    excel_filename = None
    print("No file uploaded. APPROVED_DRUGS_SET will not be updated from Excel.")

Please upload your Excel file containing approved drug names:


Saving approved drigs.xlsx to approved drigs (2).xlsx
User uploaded file "approved drigs (2).xlsx"


In [10]:
# @title Enter the column name containing drug names
drug_name_column = "G" # @param {type:"string"}

if excel_filename and drug_name_column:
    try:
        df_approved_drugs = pd.read_excel(excel_filename)
        if drug_name_column in df_approved_drugs.columns:
            new_approved_drugs = df_approved_drugs[drug_name_column].astype(str).str.lower().tolist()
            global APPROVED_DRUGS_SET # Declare intent to modify global variable
            APPROVED_DRUGS_SET = APPROVED_DRUGS_SET.union(set(new_approved_drugs)) # Add new drugs to existing set
            print(f"Successfully updated APPROVED_DRUGS_SET with {len(new_approved_drugs)} drugs from '{excel_filename}'.")
            print(f"Total unique approved drugs now: {len(APPROVED_DRUGS_SET)}")
        else:
            print(f"Error: Column '{drug_name_column}' not found in the uploaded Excel file.")
    except Exception as e:
        print(f"Error reading or processing Excel file: {e}")
else:
    if not excel_filename:
        print("No Excel file was uploaded to process.")
    if not drug_name_column:
        print("Please provide the column name containing drug names.")


Error: Column 'G' not found in the uploaded Excel file.


Now that `APPROVED_DRUGS_SET` has been updated, please re-run the 'Classify trials' cell (Cell 4) to apply the new drug list to the trial classification.

In [12]:
# Cell 3: Query ClinicalTrials.gov
def query_clinicaltrials(condition, page_token=None):
    params = {
        "query.cond": condition,
        "query.term": "AREA[InterventionType]DRUG",
        "filter.overallStatus": "COMPLETED,TERMINATED,ACTIVE_NOT_RECRUITING,RECRUITING,ENROLLING_BY_INVITATION,UNKNOWN",
        "fields": (
            "NCTId,BriefTitle,OverallStatus,Phase,StartDate,CompletionDate,"
            "LeadSponsorName,Condition,InterventionName,InterventionType,"
            "ArmGroupType,BriefSummary,EnrollmentCount,StudyType"
        ),
        "pageSize": 100,
        "format": "json"
    }
    if page_token:
        params["pageToken"] = page_token
    r = requests.get(CT_BASE, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def extract_study_fields(study):
    proto   = study.get("protocolSection", {})
    ident   = proto.get("identificationModule", {})
    status  = proto.get("statusModule", {})
    design  = proto.get("designModule", {})
    arms    = proto.get("armsInterventionsModule", {})
    desc    = proto.get("descriptionModule", {})
    sponsor = proto.get("sponsorCollaboratorsModule", {})

    nct    = ident.get("nctId", "")
    title  = ident.get("briefTitle", "")
    phase  = " / ".join(design.get("phases", []) or ["N/A"])
    start  = status.get("startDateStruct", {}).get("date", "")
    end    = status.get("completionDateStruct", {}).get("date", "")
    enroll = design.get("enrollmentInfo", {}).get("count", "")
    lead   = sponsor.get("leadSponsor", {}).get("name", "")
    summary = desc.get("briefSummary", "")[:500]

    conditions = proto.get("conditionsModule", {}).get("conditions", [])
    interventions = arms.get("interventions", [])
    drug_interventions = [
        i.get("name", "") for i in interventions
        if i.get("type", "").upper() == "DRUG"
    ]
    arm_types = list({a.get("type", "") for a in arms.get("armGroups", [])})

    return {
        "NCT_Number": nct,
        "Title": title,
        "Overall_Status": status.get("overallStatus", ""),
        "Phase": phase,
        "Start_Date": start,
        "Completion_Date": end,
        "Enrollment": enroll,
        "Lead_Sponsor": lead,
        "Conditions": "; ".join(conditions),
        "Drug_Interventions": "; ".join(drug_interventions),
        "Arm_Types": "; ".join(arm_types),
        "Brief_Summary": summary,
        "CT_Link": f"https://clinicaltrials.gov/study/{nct}"
    }

all_studies_raw = {}

for condition in tqdm(PSYCHIATRIC_CONDITIONS, desc="Querying conditions"):
    page_token = None
    pages_fetched = 0
    while True:
        try:
            data = query_clinicaltrials(condition, page_token)
        except Exception as e:
            print(f"  ERROR on '{condition}': {e}")
            break
        for s in data.get("studies", []):
            fields = extract_study_fields(s)
            nct = fields["NCT_Number"]
            if nct and nct not in all_studies_raw:
                all_studies_raw[nct] = fields
        page_token = data.get("nextPageToken")
        pages_fetched += 1
        if not page_token or pages_fetched >= 10:
            break
        time.sleep(0.4)
    time.sleep(0.3)

print(f"\nTotal unique studies retrieved: {len(all_studies_raw)}")


Querying conditions: 100%|██████████| 19/19 [00:39<00:00,  2.09s/it]


Total unique studies retrieved: 6176


In [14]:
# Cell 4: Classify trials
def classify_trial(row):
    drugs_raw     = row["Drug_Interventions"].lower()
    title_lower   = row["Title"].lower()
    summary_lower = row["Brief_Summary"].lower()
    arm_types     = row["Arm_Types"].lower()

    if not drugs_raw:
        return "EXCLUDE", "", "", "No drug interventions listed"

    for excl in EXCLUSION_TERMS:
        if excl in drugs_raw or excl in title_lower:
            return "EXCLUDE", "", "", f"Excluded intervention term: '{excl}'"

    drug_list = [d.strip() for d in row["Drug_Interventions"].split(";") if d.strip()]
    primary_found, non_primary_found_approved, non_primary_found_not_approved, uncertain_found = [], [], [], []

    for drug in drug_list:
        dl = drug.lower()
        if "placebo" in dl or "inactive" in dl:
            continue

        matched_as_primary = False
        for psych_drug in PRIMARY_PSYCH_DRUGS:
            if psych_drug in dl:
                primary_found.append(drug)
                matched_as_primary = True
                break

        if not matched_as_primary:
            if any(t in dl for t in ["cannabis","cbd","thc","cannabidiol","nabilone","dronabinol","prazosin"]):
                uncertain_found.append(drug)
            else:
                # Check if this non-primary drug is in the approved list
                is_approved = False
                for approved_drug_name in APPROVED_DRUGS_SET:
                    if approved_drug_name in dl:
                        is_approved = True
                        break

                if is_approved:
                    non_primary_found_approved.append(drug)
                else:
                    non_primary_found_not_approved.append(drug)

    setting_parts = []
    if any(kw in summary_lower for kw in ["adjunct","add-on","augment","combination"]):
        setting_parts.append("adjunctive")
    if any(kw in summary_lower for kw in ["side effect","adverse effect","weight gain","metabolic","tardive","sexual dysfunction"]):
        setting_parts.append("side-effect management")
    if any(kw in summary_lower for kw in ["inpatient","hospitali"]):
        setting_parts.append("inpatient")
    elif any(kw in summary_lower for kw in ["outpatient","community","clinic"]):
        setting_parts.append("outpatient")
    if not setting_parts:
        setting_parts.append("not specified")
    setting = "; ".join(setting_parts)

    # New exclusion logic: If any non-primary drug is not found in the APPROVED_DRUGS_SET, exclude the trial.
    if non_primary_found_not_approved:
        return "EXCLUDE", "", "", f"Contains non-approved or new drug(s) not marketed: {'; '.join(non_primary_found_not_approved)}"

    if non_primary_found_approved:
        note = ""
        if primary_found:
            note = f"Used alongside primary psychiatric drug(s): {'; '.join(primary_found)}"
        return "INCLUDE", "; ".join(non_primary_found_approved), setting, note

    if uncertain_found:
        return "UNCERTAIN", "; ".join(uncertain_found), setting, \
            "Requires manual review (cannabis-derived, prazosin, or ambiguous agent)"

    if primary_found:
        return "EXCLUDE", "", "", f"All drugs are primary psychiatric agents: {'; '.join(primary_found)}"

    return "UNCERTAIN", row["Drug_Interventions"], setting, "Could not classify — manual review required"

classified = []
for nct, row in all_studies_raw.items():
    flag, repurposed, setting, notes = classify_trial(row)
    row["Inclusion_Flag"]  = flag
    row["Repurposed_Drug"] = repurposed
    row["Setting_Context"] = setting
    row["Notes"]           = notes
    classified.append(row)

included  = [r for r in classified if r["Inclusion_Flag"] == "INCLUDE"]
uncertain = [r for r in classified if r["Inclusion_Flag"] == "UNCERTAIN"]
excluded  = [r for r in classified if r["Inclusion_Flag"] == "EXCLUDE"]

print(f"INCLUDE:   {len(included)}")
print(f"UNCERTAIN: {len(uncertain)}")
print(f"EXCLUDE:   {len(excluded)}")

INCLUDE:   163
UNCERTAIN: 168
EXCLUDE:   5845


In [15]:
# Cell 5: Query PubMed
def search_pubmed_by_nct(nct_id):
    params = {
        "db": "pubmed",
        "term": f"{nct_id}[Secondary Source ID] OR {nct_id}[Title/Abstract]",
        "retmode": "json",
        "retmax": 20
    }
    r = requests.get(PUBMED_SEARCH, params=params, timeout=20)
    r.raise_for_status()
    return r.json().get("esearchresult", {}).get("idlist", [])

def fetch_pubmed_details(pmid_list):
    if not pmid_list:
        return []
    params = {
        "db": "pubmed",
        "id": ",".join(pmid_list),
        "retmode": "xml",
        "rettype": "abstract"
    }
    r = requests.get(PUBMED_FETCH, params=params, timeout=30)
    r.raise_for_status()
    return parse_pubmed_xml(r.text)

def parse_pubmed_xml(xml_text):
    results = []
    for article in re.split(r'<PubmedArticle>', xml_text)[1:]:
        pmid = ""
        m = re.search(r'<PMID[^>]*>(\d+)</PMID>', article)
        if m:
            pmid = m.group(1)

        title_m   = re.search(r'<ArticleTitle>(.*?)</ArticleTitle>', article, re.S)
        title     = re.sub(r'<[^>]+>', '', title_m.group(1)).strip() if title_m else ""

        author_matches = re.findall(r'<LastName>(.*?)</LastName>.*?<ForeName>(.*?)</ForeName>', article, re.S)
        authors = "; ".join(f"{ln} {fn}" for ln, fn in author_matches[:6])
        if len(author_matches) > 6:
            authors += " et al."

        journal_m = re.search(r'<Title>(.*?)</Title>', article, re.S)
        journal   = re.sub(r'<[^>]+>', '', journal_m.group(1)).strip() if journal_m else ""

        year_m = re.search(r'<PubDate>.*?<Year>(\d{4})</Year>', article, re.S)
        year   = year_m.group(1) if year_m else ""
        vol_m  = re.search(r'<Volume>(.*?)</Volume>', article)
        iss_m  = re.search(r'<Issue>(.*?)</Issue>', article)
        pag_m  = re.search(r'<MedlinePgn>(.*?)</MedlinePgn>', article)
        doi_m  = re.search(r'<ArticleId IdType="doi">(.*?)</ArticleId>', article)

        vol   = vol_m.group(1) if vol_m else ""
        iss   = iss_m.group(1) if iss_m else ""
        pages = pag_m.group(1) if pag_m else ""
        doi   = doi_m.group(1).strip() if doi_m else ""

        citation = f"{authors}. {title}. {journal}. {year}"
        if vol:   citation += f";{vol}"
        if iss:   citation += f"({iss})"
        if pages: citation += f":{pages}"
        citation += "."

        results.append({
            "PMID": pmid, "Title": title, "Authors": authors,
            "Journal": journal, "Year": year, "Volume": vol,
            "Issue": iss, "Pages": pages, "DOI": doi,
            "Full_Citation": citation,
            "PubMed_Link": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""
        })
    return results

pubmed_results = []
nct_to_pmids   = {}
trials_to_query = included + uncertain

for row in tqdm(trials_to_query, desc="Querying PubMed"):
    nct = row["NCT_Number"]
    try:
        pmids = search_pubmed_by_nct(nct)
        time.sleep(0.35)
        if pmids:
            details = fetch_pubmed_details(pmids)
            nct_to_pmids[nct] = pmids
            for d in details:
                d["NCT_Number"] = nct
                pubmed_results.append(d)
        else:
            nct_to_pmids[nct] = []
        time.sleep(0.35)
    except Exception as e:
        print(f"  PubMed error for {nct}: {e}")
        nct_to_pmids[nct] = []

trials_with_pubs = sum(1 for v in nct_to_pmids.values() if v)
print(f"\nTrials with >= 1 PubMed publication: {trials_with_pubs} / {len(trials_to_query)}")
print(f"Total PubMed records found: {len(pubmed_results)}")


Querying PubMed: 100%|██████████| 331/331 [04:26<00:00,  1.24it/s]


Trials with >= 1 PubMed publication: 99 / 331
Total PubMed records found: 170


In [16]:
# Cell 6: Build Excel workbook
HEADER_FILL    = PatternFill("solid", start_color="1F4E79")
HEADER_FONT    = Font(bold=True, color="FFFFFF", name="Arial", size=10)
BODY_FONT      = Font(name="Arial", size=9)
INCLUDE_FILL   = PatternFill("solid", start_color="E2EFDA")
UNCERTAIN_FILL = PatternFill("solid", start_color="FFF2CC")
EXCLUDE_FILL   = PatternFill("solid", start_color="FCE4D6")
WRAP           = Alignment(wrap_text=True, vertical="top")

wb  = Workbook()
ws1 = wb.active
ws1.title = "ClinicalTrials.gov trials"

CT_COLUMNS = [
    ("NCT_Number",        "NCT Number",               18),
    ("Inclusion_Flag",    "Inclusion Flag",            14),
    ("Repurposed_Drug",   "Repurposed Drug",           28),
    ("Conditions",        "Psychiatric Condition",     35),
    ("Setting_Context",   "Setting / Context",         28),
    ("Overall_Status",    "Trial Status",              18),
    ("Phase",             "Phase",                     14),
    ("Title",             "Trial Title",               50),
    ("Start_Date",        "Start Date",                14),
    ("Completion_Date",   "Completion Date",           16),
    ("Enrollment",        "Enrollment (N)",            14),
    ("Lead_Sponsor",      "Lead Sponsor",              30),
    ("Drug_Interventions","All Drug Interventions",    40),
    ("CT_Link",           "ClinicalTrials.gov Link",   45),
    ("Notes",             "Notes / Uncertainty",       45),
]

for col_idx, (_, header, width) in enumerate(CT_COLUMNS, 1):
    cell = ws1.cell(row=1, column=col_idx, value=header)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws1.column_dimensions[get_column_letter(col_idx)].width = width

ws1.row_dimensions[1].height = 30
ws1.freeze_panes = "A2"

flag_fill_map = {"INCLUDE": INCLUDE_FILL, "UNCERTAIN": UNCERTAIN_FILL, "EXCLUDE": EXCLUDE_FILL}
ordered = included + uncertain + excluded

for r_idx, row in enumerate(ordered, 2):
    row_fill = flag_fill_map.get(row["Inclusion_Flag"])
    for col_idx, (field, _, _) in enumerate(CT_COLUMNS, 1):
        val  = row.get(field, "")
        cell = ws1.cell(row=r_idx, column=col_idx, value=str(val) if val != "" else "")
        cell.font      = BODY_FONT
        cell.alignment = WRAP
        if row_fill:
            cell.fill = row_fill

ws2 = wb.create_sheet("Published papers")

PUB_COLUMNS = [
    ("NCT_Number",    "NCT Number",     18),
    ("PMID",          "PubMed ID",      14),
    ("DOI",           "DOI",            35),
    ("Authors",       "Authors",        45),
    ("Year",          "Year",           10),
    ("Title",         "Paper Title",    55),
    ("Journal",       "Journal",        35),
    ("Volume",        "Volume",         10),
    ("Issue",         "Issue",          10),
    ("Pages",         "Pages",          14),
    ("Full_Citation", "Full Citation",  65),
    ("PubMed_Link",   "PubMed Link",    45),
]

for col_idx, (_, header, width) in enumerate(PUB_COLUMNS, 1):
    cell = ws2.cell(row=1, column=col_idx, value=header)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws2.column_dimensions[get_column_letter(col_idx)].width = width

ws2.row_dimensions[1].height = 30
ws2.freeze_panes = "A2"

for r_idx, pub in enumerate(pubmed_results, 2):
    for col_idx, (field, _, _) in enumerate(PUB_COLUMNS, 1):
        val  = pub.get(field, "")
        cell = ws2.cell(row=r_idx, column=col_idx, value=str(val) if val else "")
        cell.font      = BODY_FONT
        cell.alignment = WRAP

EXCEL_PATH = "drug_repurposing_trials.xlsx"
wb.save(EXCEL_PATH)
print(f"Excel saved: {EXCEL_PATH}")
print(f"  Sheet 1 rows (trials): {len(ordered)}")
print(f"  Sheet 2 rows (papers): {len(pubmed_results)}")


Excel saved: drug_repurposing_trials.xlsx
  Sheet 1 rows (trials): 6176
  Sheet 2 rows (papers): 170


In [17]:
# Cell 7: Export RIS files
def build_ct_ris(trial):
    yr = trial.get("Start_Date", "")[:4]
    lines = ["TY  - DATA", f"TI  - {trial['Title']}", f"AN  - {trial['NCT_Number']}"]
    if yr:
        lines.append(f"PY  - {yr}")
    lines += [
        f"UR  - {trial['CT_Link']}",
        f"ST  - {trial['Overall_Status']}",
        f"AB  - {trial['Brief_Summary']}",
        f"KW  - Drug repurposing",
        f"KW  - Psychiatry",
        f"KW  - {trial['Repurposed_Drug']}",
        f"KW  - {trial['Conditions']}",
        "DB  - ClinicalTrials.gov",
        "ER  - ", ""
    ]
    return "\n".join(lines)

def build_pubmed_ris(pub):
    lines = ["TY  - JOUR"]
    if pub.get("Title"):   lines.append(f"TI  - {pub['Title']}")
    for author in pub.get("Authors", "").split("; "):
        if author.strip():
            lines.append(f"AU  - {author.strip()}")
    if pub.get("Journal"): lines.append(f"JO  - {pub['Journal']}")
    if pub.get("Year"):    lines.append(f"PY  - {pub['Year']}")
    if pub.get("Volume"):  lines.append(f"VL  - {pub['Volume']}")
    if pub.get("Issue"):   lines.append(f"IS  - {pub['Issue']}")
    if pub.get("Pages"):   lines.append(f"SP  - {pub['Pages']}")
    if pub.get("DOI"):     lines.append(f"DO  - {pub['DOI']}")
    if pub.get("PMID"):    lines.append(f"AN  - {pub['PMID']}")
    if pub.get("PubMed_Link"): lines.append(f"UR  - {pub['PubMed_Link']}")
    if pub.get("NCT_Number"):  lines.append(f"KW  - {pub['NCT_Number']}")
    lines += ["KW  - Drug repurposing", "KW  - Psychiatry", "ER  - ", ""]
    return "\n".join(lines)

CT_RIS_PATH  = "clinicaltrials_references.ris"
PUB_RIS_PATH = "pubmed_references.ris"

with open(CT_RIS_PATH, "w", encoding="utf-8") as f:
    for trial in ordered:
        f.write(build_ct_ris(trial))

with open(PUB_RIS_PATH, "w", encoding="utf-8") as f:
    for pub in pubmed_results:
        f.write(build_pubmed_ris(pub))

print(f"RIS files saved:")
print(f"  {CT_RIS_PATH}  ({len(ordered)} records)")
print(f"  {PUB_RIS_PATH} ({len(pubmed_results)} records)")


RIS files saved:
  clinicaltrials_references.ris  (6176 records)
  pubmed_references.ris (170 records)


In [18]:
# Cell 8: Download files from Colab
from google.colab import files
for path in [EXCEL_PATH, CT_RIS_PATH, PUB_RIS_PATH]:
    files.download(path)
    print(f'Download triggered: {path}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered: drug_repurposing_trials.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered: clinicaltrials_references.ris


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered: pubmed_references.ris


In [19]:
# Cell 9: Print summary report
print("=" * 60)
print("SEARCH SUMMARY REPORT")
print(f"Run date: {datetime.today().strftime('%Y-%m-%d')}")
print("=" * 60)
print(f"Conditions searched:            {len(PSYCHIATRIC_CONDITIONS)}")
print(f"Total unique trials retrieved:  {len(all_studies_raw)}")
print()
print(f"INCLUDE  (repurposed drug):     {len(included)}")
print(f"UNCERTAIN (manual review):      {len(uncertain)}")
print(f"EXCLUDE  (primary psych drug):  {len(excluded)}")
print()
print(f"Trials with >=1 PubMed paper:  {trials_with_pubs}")
print(f"Total PubMed records:           {len(pubmed_results)}")
print()
print("KEY LIMITATIONS")
print("-" * 60)
print("1. No native repurposing filter in ClinicalTrials.gov.")
print("   Classification is heuristic -- manual review required.")
print("2. Drug exclusion list covers common primary psychiatric")
print("   agents but is not exhaustive.")
print("3. PubMed NCT linkage is incomplete. Published results")
print("   not indexed under the NCT number will be missed.")
print("4. Search capped at 1000 results/condition (10 pages).")
print("   High-volume conditions may be truncated.")
print("5. Trials without a listed drug intervention type are")
print("   excluded automatically -- some may warrant inclusion.")


SEARCH SUMMARY REPORT
Run date: 2026-05-10
Conditions searched:            19
Total unique trials retrieved:  6176

INCLUDE  (repurposed drug):     163
UNCERTAIN (manual review):      168
EXCLUDE  (primary psych drug):  5845

Trials with >=1 PubMed paper:  99
Total PubMed records:           170

KEY LIMITATIONS
------------------------------------------------------------
1. No native repurposing filter in ClinicalTrials.gov.
   Classification is heuristic -- manual review required.
2. Drug exclusion list covers common primary psychiatric
   agents but is not exhaustive.
3. PubMed NCT linkage is incomplete. Published results
   not indexed under the NCT number will be missed.
4. Search capped at 1000 results/condition (10 pages).
   High-volume conditions may be truncated.
5. Trials without a listed drug intervention type are
   excluded automatically -- some may warrant inclusion.
